# Notebook 3 — Exploratory Data Analysis (EDA)

Exploratory analysis of the final dataset to understand patterns before modeling.

## Sections
1. Dataset summary statistics
2. Class balance check
3. Temporal patterns (flood rate per year/month)
4. Rainfall distribution
5. Feature correlation heatmap
6. Flood rate by distance to river
7. Flood rate by elevation

## 1. Imports and Load Data

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

sns.set_style("whitegrid")
plt.rcParams["figure.dpi"] = 100

df = pd.read_csv("../data/processed/dhemaji_flood_FINAL.csv")
df["date"] = pd.to_datetime(df["date"])
df["year"] = df["date"].dt.year
df["month"] = df["date"].dt.month

print("Dataset shape:", df.shape)
print("Date range:", df["date"].min(), "to", df["date"].max())

## 2. Dataset Summary

In [ ]:
print("="*50)
print("DATASET SUMMARY")
print("="*50)
print(f"Total rows:           {len(df):,}")
print(f"Unique grid cells:    {df[['grid_lat','grid_lon']].drop_duplicates().shape[0]:,}")
print(f"Unique dates:         {df['date'].nunique()}")
print(f"Years covered:        {sorted(df['year'].unique())}")
print(f"Total features:       {df.shape[1]}")
print(f"Missing values:       {df.isna().sum().sum()}")

## 3. Class Balance

In [ ]:
print("Class distribution:")
print(df["flood_label"].value_counts())
print("\nClass proportions:")
print(df["flood_label"].value_counts(normalize=True).round(3))

# Visualize
fig, ax = plt.subplots(figsize=(8,5))
df["flood_label"].value_counts().plot(
    kind="bar", color=["steelblue","coral"], ax=ax
)
ax.set_title("Class Distribution — Flood vs Non-Flood")
ax.set_xlabel("Flood Label (0=No Flood, 1=Flood)")
ax.set_ylabel("Count")
ax.set_xticklabels(["No Flood","Flood"], rotation=0)
plt.tight_layout()
plt.savefig("../figures/class_distribution.png", dpi=150)
plt.show()

## 4. Flood Rate Per Year

Check if flooding patterns are consistent across years.

In [ ]:
flood_by_year = df.groupby("year")["flood_label"].mean()
print("Flood rate per year:")
print(flood_by_year.round(3))

fig, ax = plt.subplots(figsize=(10,5))
flood_by_year.plot(
    kind="bar", color="steelblue", ax=ax
)
ax.set_title("Flood Rate by Year")
ax.set_ylabel("Proportion of Cells Flooded")
ax.set_xlabel("Year")
ax.set_xticklabels(flood_by_year.index, rotation=0)
plt.tight_layout()
plt.savefig("../figures/flood_rate_by_year.png", dpi=150)
plt.show()

## 5. Flood Rate Per Month

In [ ]:
flood_by_month = df.groupby("month")["flood_label"].mean()
print("Flood rate per month:")
print(flood_by_month.round(3))

fig, ax = plt.subplots(figsize=(10,5))
flood_by_month.plot(
    kind="bar", color="coral", ax=ax
)
ax.set_title("Flood Rate by Month")
ax.set_ylabel("Proportion of Cells Flooded")
ax.set_xlabel("Month")
month_labels = ["Jun","Jul","Aug","Sep"]
ax.set_xticklabels(month_labels, rotation=0)
plt.tight_layout()
plt.savefig("../figures/flood_rate_by_month.png", dpi=150)
plt.show()

## 6. Rainfall Distribution

In [ ]:
fig, ax = plt.subplots(figsize=(12,5))
ax.hist(df["rainfall_mm"], bins=50, color="steelblue", edgecolor="black")
ax.set_title("Distribution of Daily Rainfall")
ax.set_xlabel("Rainfall (mm)")
ax.set_ylabel("Frequency")
plt.tight_layout()
plt.savefig("../figures/rainfall_distribution.png", dpi=150)
plt.show()

print("Rainfall statistics:")
print(df["rainfall_mm"].describe().round(2))

## 7. Feature Correlation Heatmap

In [ ]:
corr_features = [
    "rainfall_mm","rain_3day","rain_5day",
    "runoff_sum",
    "elevation","slope","tree_cover",
    "dist_to_major_river",
    "flood_label"
]

corr = df[corr_features].corr()

fig, ax = plt.subplots(figsize=(11,9))
sns.heatmap(
    corr, annot=True, fmt=".2f",
    cmap="coolwarm", center=0,
    square=True, ax=ax
)
ax.set_title("Feature Correlation Heatmap")
plt.tight_layout()
plt.savefig("../figures/correlation_heatmap.png", dpi=150)
plt.show()

## 8. Flood Rate by Distance to Major River

The single most important spatial pattern.

In [ ]:
df["dist_quartile"] = pd.qcut(
    df["dist_to_major_river"],
    q=4,
    labels=["Closest","Near","Far","Farthest"]
)

flood_by_dist = df.groupby("dist_quartile", observed=True)["flood_label"].mean()
print("Flood rate by distance to major river:")
print(flood_by_dist.round(3))

fig, ax = plt.subplots(figsize=(10,5))
flood_by_dist.plot(
    kind="bar", color="steelblue", ax=ax
)
ax.set_title("Flood Rate by Distance to Major River")
ax.set_ylabel("Flood Rate")
ax.set_xlabel("Distance Quartile")
plt.xticks(rotation=0)
plt.tight_layout()
plt.savefig("../figures/flood_by_distance.png", dpi=150)
plt.show()

## 9. Flood Rate by Elevation

In [ ]:
df["elev_quartile"] = pd.qcut(
    df["elevation"],
    q=4,
    labels=["Lowest","Low","High","Highest"]
)

flood_by_elev = df.groupby("elev_quartile", observed=True)["flood_label"].mean()
print("Flood rate by elevation:")
print(flood_by_elev.round(3))

fig, ax = plt.subplots(figsize=(10,5))
flood_by_elev.plot(
    kind="bar", color="coral", ax=ax
)
ax.set_title("Flood Rate by Elevation")
ax.set_ylabel("Flood Rate")
ax.set_xlabel("Elevation Quartile")
plt.xticks(rotation=0)
plt.tight_layout()
plt.savefig("../figures/flood_by_elevation.png", dpi=150)
plt.show()

## 10. Key Findings

- Flood rate is ~5-6% — highly imbalanced
- Flooding is consistent across years (2019-2024)
- Rainfall distribution is highly right-skewed with many zero values
- Distance to major river is the strongest predictor — closest cells flood 300x more than farthest
- Elevation matters but less than river proximity
